# Autoencoder: Anomalieerkennung (Praesentation)

Dieses Notebook basiert auf den Dateien `autoencoder_model.py`, `autoencoder_train.py`, `autoencoder_test.py` und `autoencoder_gridsearch.py`.

## 1. Setup und Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from itertools import product
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parents[1]
SRC_DIR = PROJECT_ROOT / "01_src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_utils import read_data, sensor_cols
from autoencoder_model import fit_pipeline, predict_pipeline

## 2. Daten laden und vorbereiten

In [ ]:
train_path = PROJECT_ROOT / "02_data" / "train" / "train_FD001.txt"
test_path = PROJECT_ROOT / "02_data" / "test" / "test_FD001.txt"
rul_path = PROJECT_ROOT / "02_data" / "RUL" / "RUL_FD001.txt"

train_df = read_data(train_path)
test_df = read_data(test_path)
rul_df = pd.read_csv(rul_path, header=None, names=["RUL"])
rul_df["unit_id"] = rul_df.index + 1

threshold_std = 0.01
useful_sensors = [col for col in sensor_cols if train_df[col].std() > threshold_std]

max_cycles = train_df.groupby("unit_id")["cycles"].max().reset_index()
max_cycles.columns = ["unit_id", "max_cycle"]
train_df = train_df.merge(max_cycles, on="unit_id")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycles"]

scaler = StandardScaler()
train_df[useful_sensors] = scaler.fit_transform(train_df[useful_sensors])
test_df[useful_sensors] = scaler.transform(test_df[useful_sensors])

healthy_filter = train_df["cycles"] <= train_df["max_cycle"] * 0.3
healthy_df = train_df[healthy_filter]

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Anzahl Sensorsignale (nach Filter): {len(useful_sensors)}")

## 3. Trainingslauf (wie in autoencoder_train.py)

In [ ]:
window_size = 20
hidden_size = 32
lr = 0.001
epochs = 50
threshold = 0.3

train_results_df, model = fit_pipeline(
    train_df, healthy_df, useful_sensors,
    window_size, hidden_size, lr, epochs, threshold
)

y_true_train = (train_results_df["RUL"] <= 30).astype(int)
y_pred_train = train_results_df["anomaly_flag"].astype(int)

print(f"Precision: {precision_score(y_true_train, y_pred_train):.4f}")
print(f"Recall:    {recall_score(y_true_train, y_pred_train):.4f}")
print(f"F1:        {f1_score(y_true_train, y_pred_train):.4f}")

In [ ]:
first_anomaly = (
    train_results_df[train_results_df["anomaly_flag"] == 1]
    .groupby("unit_id")["RUL"]
    .max()
)
first_anomaly.describe()

## 4. Testlauf (wie in autoencoder_test.py)

In [ ]:
test_results_df = predict_pipeline(test_df, model, useful_sensors, window_size, threshold)
test_results_df = test_results_df.merge(rul_df, on="unit_id", how="left")

units_with_alarm = test_results_df[test_results_df["anomaly_flag"] == 1]["unit_id"].nunique()
alarmed_units = test_results_df[test_results_df["anomaly_flag"] == 1]["unit_id"].unique()

print(f"Units mit Alarm: {units_with_alarm} / 100")
print("RUL der Units MIT Alarm:")
print(rul_df[rul_df["unit_id"].isin(alarmed_units)]["RUL"].describe())
print("RUL der Units OHNE Alarm:")
print(rul_df[~rul_df["unit_id"].isin(alarmed_units)]["RUL"].describe())

## 5. Gridsearch (wie in autoencoder_gridsearch.py)

In [ ]:
param_grid = {
    "window_size": [20, 30],
    "hidden_size": [16, 32],
    "lr": [0.001, 0.0005],
    "epochs": [30, 50],
    "threshold": [0.2, 0.3, 0.4],
}

search_results = []
best_result = None

for window_size_, hidden_size_, lr_, epochs_, threshold_ in product(
    param_grid["window_size"],
    param_grid["hidden_size"],
    param_grid["lr"],
    param_grid["epochs"],
    param_grid["threshold"],
):
    candidate_df, _ = fit_pipeline(
        train_df, healthy_df, useful_sensors,
        window_size_, hidden_size_, lr_, epochs_, threshold_
    )
    if candidate_df is None:
        continue

    y_true = (candidate_df["RUL"] <= 30).astype(int)
    y_pred = candidate_df["anomaly_flag"].astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    row = {
        "window_size": window_size_,
        "hidden_size": hidden_size_,
        "lr": lr_,
        "epochs": epochs_,
        "threshold": threshold_,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    search_results.append(row)

    if best_result is None or (f1, recall, precision) > (best_result["f1"], best_result["recall"], best_result["precision"]):
        best_result = row

grid_df = pd.DataFrame(search_results).sort_values(by=["f1", "recall", "precision"], ascending=False)
grid_df.head(10)

In [ ]:
best_result

## 6. Folien-Kernaussagen

- Der Autoencoder lernt auf fruehen (gesunden) Zyklen ein Rekonstruktionsmuster.
- Ein Alarm wird gesetzt, wenn der Rekonstruktionsfehler den Schwellwert ueberschreitet.
- Die Gridsearch zeigt den Trade-off zwischen Precision und Recall.
- Die Aussagekraft fuer die Wartung ergibt sich aus der RUL-Verteilung der alarmierten Units.